# Primary Questions Analysis

## Primary Questions and Analysis Plan:

From the Analysis Framework, the following questions were suggested as the primary concern:

- What are great predictors of good sleep duration and sleep quality? 
- Are there patterns amongst individuals who have sleep disorders vs individuals who do not? 
- Is the type of sleep disorder reflected in the other variables/ can we differentiate between sleep disorders using the data provided?

Thus, each section will use various statistical methods to find the answers to these questions.

In [47]:
# setting up the notebook
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from xml.parsers.expat import model
from statsmodels.othermod.betareg import BetaModel

# data read in
df = pd.read_csv('./data/modified_sleep_health_and_lifestyle_dataset.csv')

## Section 1: What are the best predictors for sleep duration and quality?

This section will contain two parts: feature engineering and predictive modeling.

For feature engineering, both sleep duration and sleep quality have ranges for which they are contained within. Sleep quality is assumed to be on a scale from 1-10 and recorded as an integer. On the other hand, sleep duration being greater than 24 (hours/day) is not possible. One solution would be to turn these variables into binary (above some standard). A second option would be to scale the values to a 0-1 range (beta regression). Yet, another would be to merely keep them as is and interpret as necessary.

For this section, the first two methods will be used to both explore thresholds and retain as much information as possible. For the thresholds, sleep quality was set to 5 as being the middle ground. For sleep duration, the threshold was set to 7 as the average adult needs 7 hours of sleep. Additionally, the range for sleep duration will be assumed to be between 0-15 hours as 15 hours is the maximum number of hours an adult can naturally sleep for.

In [48]:
# sleep_quality to binary:
df['sleep_quality_binary'] = df['sleep_quality'].apply(lambda x: 1 if x >= 5 else 0)

# sleep duration to binary:
df['sleep_duration_binary'] = df['sleep_duration'].apply(lambda x: 1 if x >= 7 else 0)

# sleep quality rescaled to 0-1: 
df['sleep_quality_rescaled'] = (df['sleep_quality'] - 1)/9

# sleep duration rescaled to 0-1:
df['sleep_duration_rescaled'] = df['sleep_duration']/15


Additionally, to build some of our models, we need to one-hot-encode our categorical variables.

In [49]:
df['sleep_disorder'].fillna('N/A', inplace=True)
encoded_df = pd.get_dummies(df, columns=['gender', 'occupation', 'bmi_category', 'sleep_disorder'], drop_first=True, dtype=int)
encoded_df.head()

,person_id,age,sleep_duration,sleep_quality,physical_activity_level,stress_level,heart_rate,daily_steps,systolic,diastolic,...,occupation_Manager,occupation_Nurse,occupation_Sales Representative,occupation_Salesperson,occupation_Scientist,occupation_Software Engineer,occupation_Teacher,bmi_category_Overweight,sleep_disorder_N/A,sleep_disorder_Sleep Apnea
0,1,27,6.1,6,42,6,77,4200,126.0,83.0,...,0,0,0,0,0,1,0,1,1,0
1,2,28,6.2,6,60,8,75,10000,125.0,80.0,...,0,0,0,0,0,0,0,0,1,0
2,4,28,5.9,4,30,8,85,3000,140.0,90.0,...,0,0,1,0,0,0,0,1,0,1
3,6,28,5.9,4,30,8,85,3000,140.0,90.0,...,0,0,0,0,0,1,0,1,0,0
4,7,29,6.3,6,40,7,82,3500,140.0,90.0,...,0,0,0,0,0,0,1,1,0,0


Before any regression, the dataset will first be split into train and test datasets with a 80/20 split.

In [50]:
# binary train test split:
X_quality = encoded_df.drop(columns=['sleep_quality','sleep_quality_binary', 'sleep_quality_rescaled'])
X_duration = encoded_df.drop(columns=['sleep_duration','sleep_duration_binary', 'sleep_duration_rescaled'])
y_quality_binary = encoded_df['sleep_quality_binary']
y_duration_binary = encoded_df['sleep_duration_binary']
X_train_qb, X_test_qb, y_train_qb, y_test_qb = train_test_split(X_quality, y_quality_binary, test_size=0.2, random_state=10)
X_train_db, X_test_db, y_train_db, y_test_db = train_test_split(X_duration, y_duration_binary, test_size=0.2, random_state=10)

# rescaled train test split:
y_quality_rescaled = encoded_df['sleep_quality_rescaled']
y_duration_rescaled = encoded_df['sleep_duration_rescaled']
X_train_qr, X_test_qr, y_train_qr, y_test_qr = train_test_split(X_quality, y_quality_rescaled, test_size=0.2, random_state=10)
X_train_dr, X_test_dr, y_train_dr, y_test_dr = train_test_split(X_duration, y_duration_rescaled, test_size=0.2, random_state=10)

### Rescaled Regression

This subsection will be for examining the rescaled variables and all conclusions derived. While values in theory could be either exactly 0 or 1 for both variables of interest, our EDA suggested that it would be highly unlikely to have observed individuals sleep 0 or 15 hours or individuals who rate their sleep quality as 1 or 10. This allows for the use of beta regression. Alternative models to start with as well could include fractional logistic regression and fractional probit regression. My hypothesis is that due to the way these variables are typically distributed, the probit link function/ normal distribution assumption will work better to predict the values than the logistic regression.



#### Sleep Duration: Starting Models

Three models will be built: fractional probit, fractional logit, and beta regression to start.

In [51]:
# fractional probit

X_train_dr = sm.add_constant(X_train_dr) #intercept
X_test_dr = sm.add_constant(X_test_dr, has_constant='add')
model_fp = sm.GLM(y_train_dr, X_train_dr, family=sm.families.Binomial(link=sm.genmod.families.links.probit()))
results_fp = model_fp.fit()
print(results_fp.summary())

                    Generalized Linear Model Regression Results                    
Dep. Variable:     sleep_duration_rescaled   No. Observations:                  105
Model:                                 GLM   Df Residuals:                       81
Model Family:                     Binomial   Df Model:                           23
Link Function:                      probit   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:                -47.084
Date:                     Sat, 02 May 2026   Deviance:                      0.10803
Time:                             11:56:07   Pearson chi2:                    0.108
No. Iterations:                          3   Pseudo R-squ. (CS):           0.009991
Covariance Type:                 nonrobust                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------

c:\Users\saval489\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The probit link alias is deprecated. Use Probit instead. The probit link alias will be removed after the 0.15.0 release.
  warnings.warn(


With the p-values all being close to 1 and the pseudo-Rsquared being very low, the summary results seem to indicate that our fractional probit model does not perform well. Thus, for now, there will be no further evaluation of the model with k-fold CV. Additionally, the fractional logit will be skipped for now due to the similarity of the two models and the expected similarity in performances.

In [53]:


kfolds = 5
X_train_dr['fold'] = pd.qcut(X_train_dr.index, q=kfolds, labels=False)

for fold in range(kfolds):
    train_fold = X_train_dr[X_train_dr['fold'] != fold].drop(columns=['fold'])
    test_fold = X_train_dr[X_train_dr['fold'] == fold].drop(columns=['fold'])
    y_train_fold = y_train_dr[train_fold.index]
    y_test_fold = y_train_dr[test_fold.index]
    
    beta_model = BetaModel(y_train_fold, train_fold)
    beta_results = beta_model.fit()
    print(f"Fold {fold} results:")
    print(beta_results.summary())




Fold 0 results:
                                 BetaModel Results                                 
Dep. Variable:     sleep_duration_rescaled   Log-Likelihood:                 257.40
Model:                           BetaModel   AIC:                            -460.8
Method:                 Maximum Likelihood   BIC:                            -395.2
Date:                     Sat, 02 May 2026                                         
Time:                             12:32:42                                         
No. Observations:                       84                                         
Df Residuals:                           57                                         
Df Model:                               25                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const                       

c:\Users\saval489\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
c:\Users\saval489\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
c:\Users\saval489\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
c:\Users\saval489\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no 